In [22]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import scipy.stats as stats

np.random.seed(42)

###  Генерация выборки
Создаем матрицу независимых факторов $X$ из распределения $R(-1, 1)$. Формируем матрицу наблюдений $\Psi$ и генерируем отклик $Y$ по истинной линейной модели с $\epsilon \sim N(0, 1.5^2)$.

In [52]:
X = np.random.uniform(-1, 1, size=(50, 5))
epsilon = np.random.normal(0, 1.5, size=50)

# добавляем слева столбец из единиц (для beta_0)
Psi = sm.add_constant(X)

true_beta = np.array([2, 3, -2, 1, 1, -1])
# моделируем отклик
Y = Psi @ true_beta + epsilon

column_names = ['const', 'x1', 'x2', 'x3', 'x4', 'x5']
df = pd.DataFrame(Psi, columns=column_names)
df['Y'] = Y

display(df.head())

# проверка парных корреляций
print("\n Матрица парных корреляций ")
display(df.corr().round(3))

,const,x1,x2,x3,x4,x5,Y
0,1.0,0.737131,0.873186,-0.056038,-0.853442,0.730312,0.770245
1,1.0,-0.993639,0.503976,0.093500,0.918517,-0.566474,-2.256552
2,1.0,-0.557562,-0.084556,-0.693834,0.217171,0.369126,0.955931
3,1.0,0.838387,-0.614649,0.559904,0.215350,-0.439773,8.160239
4,1.0,0.168442,0.105073,-0.325320,0.188519,0.799579,-1.857428



 Матрица парных корреляций 


,const,x1,x2,x3,x4,x5,Y
const,NaN,NaN,NaN,NaN,NaN,NaN,NaN
x1,NaN,1.000,-0.030,0.089,-0.221,-0.026,0.681
x2,NaN,-0.030,1.000,0.041,0.086,0.095,-0.479
x3,NaN,0.089,0.041,1.000,-0.197,-0.194,0.195
x4,NaN,-0.221,0.086,-0.197,1.000,0.054,-0.057
x5,NaN,-0.026,0.095,-0.194,0.054,1.000,-0.360
Y,NaN,0.681,-0.479,0.195,-0.057,-0.360,1.000


## Проверка на мультиколлинеарность

**Мультиколлинеарность** возникает, когда факторы $\xi_1, \dots, \xi_k$ коррелируют между собой. В случае сильной линейной зависимости определитель информационной матрицы обращается в ноль ($\det F = 0$). Матрица Фишера $F = \Psi^T \Psi$ становится вырожденной, что делает невозможным нахождение вектора оценок.

Способы выявить мультиколл:

1. **Анализ матрицы парных корреляций.**
2. **Определитель матрицы Фишера:**  $\det F \neq 0$.
3. **Вспомогательные регрессии:** Построение регрессионной модели каждого фактора на все остальные факторы:
   $$ \xi_i = b_1 + b_2\xi_1 + \dots + b_k\xi_k + \epsilon_i  $$
    Если коэффициент детерминации такой вспомогательной модели $R^2 > 0.7$, то данный фактор полностью определяется остальными, и его необходимо отбросить.

In [24]:
print("1. R^2 вспомогательных регрессий:")

for i in range(1, 6):
    # выбираем i-й фактор
    y_vspom = Psi[:, i]

    # убираем этот фактор из матрицы
    X_vspom = np.delete(Psi, i, axis=1)

    # фактор через остальные факторы
    model_vspom = sm.OLS(y_vspom, X_vspom).fit()

    print(f"R^2 для x{i} через остальные: {model_vspom.rsquared:.4f}")

print("\n2. Определитель матрицы F:")

F = Psi.T @ Psi
det_F = np.linalg.det(F)

print(f"det(F) = {det_F:.2e}")

# определитель не равен нулю и все R^2 меньше 0.7
if det_F > 1e-5 and all(sm.OLS(Psi[:, i], np.delete(Psi, i, axis=1)).fit().rsquared < 0.7 for i in range(1, 6)):
    print("\n Мультиколлинеарности нет.")

1. R^2 вспомогательных регрессий:
R^2 для x1 через остальные: 0.1172
R^2 для x2 через остальные: 0.1139
R^2 для x3 через остальные: 0.1429
R^2 для x4 через остальные: 0.0461
R^2 для x5 через остальные: 0.1000

2. Определитель матрицы F:
det(F) = 6.28e+07

 Мультиколлинеарности нет.


## Вычисление оценок и проверка их значимости

Строим модель множественной линейной регрессии в векторно-матричном виде:
$$ Y = \Psi\beta + \epsilon $$
где $\epsilon \sim N(0, \sigma^2 E)$ — вектор случайных независимых ошибок.

1. По МНК вычисляем вектор оценок $\tilde{\beta}$:
$$ \tilde{\beta} = F^{-1}\Psi^T Y $$

2. Для каждого коэффициента проверяется $H_0: \beta_i = 0$ против $H_1: \beta_i \neq 0$.
Статистика критерия базируется на распределении Стьюдента:
$$ \Delta = \frac{\tilde{\beta}_i}{\sqrt{RSS \cdot F_{ii}^{-1}}} \sqrt{n-p} \sim t(n-p) $$
Уровень значимости $\alpha = 0.05$. Если для коэффициента выполняется $p\text{-value} < \alpha$, нулевая гипотеза отвергается, и коэффициент признается статистически значимым.

In [25]:
# МНК-модель и вычисляем результаты
model = sm.OLS(Y, Psi)
results = model.fit()

summary_df = pd.DataFrame({
    'β': true_beta,
    'β̃': results.params,
    'sqrt(RSS/(n-p) * (F⁻¹)ii)': results.bse,
    'Δ̃': results.tvalues,
    'p-value': results.pvalues
}, index=['const (β0)', 'β1', 'β2', 'β3', 'β4', 'β5'])

alpha = 0.05
summary_df['Вывод'] = summary_df['p-value'].apply(
    lambda p: 'Значим (H0 отвергнута)' if p < alpha else 'Незначим (H0 принята)'
)

display(summary_df.round(4))

,β,β̃,sqrt(RSS/(n-p) * (F⁻¹)ii),Δ̃,p-value,Вывод
const (β0),2,2.0699,0.2467,8.3904,0.0000,Значим (H0 отвергнута)
β1,3,2.7552,0.4359,6.3208,0.0000,Значим (H0 отвергнута)
β2,-2,-2.1409,0.4471,-4.7884,0.0000,Значим (H0 отвергнута)
β3,1,1.0096,0.4362,2.3144,0.0254,Значим (H0 отвергнута)
β4,1,0.7684,0.4106,1.8711,0.0680,Незначим (H0 принята)
β5,-1,-1.2105,0.4354,-2.7800,0.0080,Значим (H0 отвергнута)


## Определение коэффициента детерминации и проверка его значимости
Проверяем гипотезу об адекватности регрессии в целом. Сравниваем полную модель $H_1: \eta = \Psi\beta + \varepsilon$ с наипростейшей $H_0: \eta = \beta_1 + \varepsilon_1$.

$R^2$ показывает долю дисперсии отклика, объясненную регрессией. Статистика критерия (Фишера):
$$ \Delta = \frac{R^2}{1 - R^2} \cdot \frac{n - p}{p - 1} \sim F(p-1, n-p) $$

*   **$p\text{-value}$** $= P(\Delta > \tilde{\Delta} \mid H_0)$. Уровень значимости $\alpha = 0.05$.
Если $p\text{-value} < \alpha$, нулевая гипотеза отвергается.

In [26]:
# извлекаем значения из уже обученной модели
r_squared = results.rsquared
delta_tilde = results.fvalue      # статистика Фишера (Δ̃)
p_value_f = results.f_pvalue      # p-value для F-критерия

print(f"R² = {r_squared:.4f}")
print(f"Δ̃ = {delta_tilde:.4f}")
print(f"p-value = {p_value_f:.4e}\n")

alpha = 0.05
if p_value_f < alpha:
    print(f"Итог: p-value < {alpha} ")
    print("Регрессия значима.")
else:
    print(f"Итог: p-value >= {alpha} ")
    print("Регрессия незначима, факторы не описывают поведение отклика.")

R² = 0.6264
Δ̃ = 14.7536
p-value = 1.7034e-08

Вывод: p-value < 0.05 
Регрессия значима.


## Прогноз в точке и 95% доверительный интервал

Для заданной точки $x_k = 0$ вектор факторов имеет вид $\Psi_0 = (1, 0, 0, 0, 0, 0)$.
Точечный прогноз в этой точке:
$$ y_0 = \Psi_0 \tilde{\beta} = \tilde{\beta}_0 $$

Нам нужно построить доверительный интервал для истинного значения отклика $\eta_0$:
$$ \eta_0 = \Psi_0 \beta + \epsilon_0, \quad \epsilon_0 \sim N(0, \sigma^2) $$
$$ \eta_0 \sim N(\Psi_0 \beta, \sigma^2) $$

Разность между прогнозом и истинным значением:
$$ y_0 - \eta_0 \sim N(0, \sigma^2(1 + \Psi_0 F^{-1} \Psi_0^T)) $$

Нормируем и переходим к распределению Стьюдента:
$$ \frac{y_0 - \eta_0}{\delta} \sim t(n-p), \quad \text{где } \delta = \frac{\sqrt{RSS} \sqrt{1 + \Psi_0 F^{-1} \Psi_0^T}}{\sqrt{n-p}} $$

$\gamma = 0.95$. Границы доверительного интервала:
$$ t_{\frac{1-\gamma}{2}}(n-p) < \frac{y_0 - \eta_0}{\delta} < t_{\frac{1+\gamma}{2}}(n-p) $$

Итоговый доверительный интервал:
$$ y_0 - \delta t_{\frac{1+\gamma}{2}}(n-p) < \eta_0 < y_0 - \delta t_{\frac{1-\gamma}{2}}(n-p) $$

Так как $\Psi_0 = (1, 0, \dots, 0)$, матричное произведение $\Psi_0 F^{-1} \Psi_0^T$ равно первому диагональному элементу $(F^{-1})_{11}$.


In [53]:
# точечный прогноз y_0
y_0 = results.params[0]
print(f" Прогноз в точке x_k=0 = {y_0:.4f}")

rss = results.ssr
n_p = results.df_resid
F_inv_11 = results.normalized_cov_params[0, 0]

gamma = 0.95
t_right = stats.t.ppf((1 + gamma) / 2, n_p)
t_left = stats.t.ppf((1 - gamma) / 2, n_p)

delta_val = (np.sqrt(rss) * np.sqrt(1 + F_inv_11)) / np.sqrt(n_p)

print(f" t_left = {t_left:.4f}, t_right = {t_right:.4f}\n")

lower_bound = y_0 - delta_val * t_right
upper_bound = y_0 - delta_val * t_left

print(f"{gamma*100:.0f}% доверительный интервал для η_0:")
print(f"{lower_bound:.4f} < η_0 < {upper_bound:.4f}")

 Прогноз в точке x_k=0 = 2.0699
 t_left = -2.0154, t_right = 2.0154

95% доверительный интервал для η_0:
-1.4232 < η_0 < 5.5630


## Проверка предположения о независимости ошибок измерения
$H_0$: ошибки независимы и одинаково распределены.
$H_1$: ошибки зависимы.

Используем критерий на основе числа инверсий $I_n$ в выборке остатков $e_i$.

Статистика критерия:
$$ \Delta = \frac{I_n - \frac{n(n-1)}{4}}{\sqrt{\frac{n^3}{36}}} \approx N(0, 1) $$

$$ p\text{-value} = 2 \cdot P(\Delta \ge |\tilde{\Delta}| \mid H_0) $$
Уровень значимости $\alpha = 0.05$.

In [ ]:
# извлекаем остатки регрессии
e = np.array(results.resid)
n = len(e)

# число инверсий I_n
I_n = sum(1 for i in range(n) for j in range(i + 1, n) if e[i] > e[j])
E_I = n * (n - 1) / 4
D_I = (n**3) / 36

# вычисляем статистику критерия Δ
delta = (I_n - E_I) / np.sqrt(D_I)

p_value = 2 * stats.norm.sf(np.abs(delta))

print(f"n = {n}")
print(f"I_n = {I_n}")
print(f"Δ = {delta:.4f}")
print(f"p-value = {p_value:.4f}\n")

alpha = 0.05
if p_value < alpha:
    print(f"Итог: p-value < {alpha}")
    print("H₀ отвергается. Ошибки зависимы.")
else:
    print(f"Итог: p-value >= {alpha}")
    print("Нет оснований отвергать H₀. Ошибки независимы.")

## Проверка гипотезы о нормальности распределения ошибок
$H_0$: $\varepsilon \sim N(0, \sigma^2)$.

Истинная дисперсия $\sigma^2$ неизвестна, оцениваем её по выборке:
$$ \tilde{\sigma}^2 = \frac{1}{n-1} \sum_{i=1}^n (e_i - \bar{e})^2 $$

 Используем параметрический бутстрап.
Наблюдаемая статистика:
$$ \tilde{\Delta} = \sqrt{n} \sup_{x \in \mathbb{R}} |\tilde{F}(x) - \Phi(x, 0, \tilde{\sigma}^2)| $$
Генерируем $N = 50000$ искусственных выборок из $N(0, \tilde{\sigma}^2)$, для каждой заново оцениваем дисперсию и считаем $\Delta^*_i$. Итоговое p-value равно доле случаев, когда $\Delta^*_i \ge \tilde{\Delta}$. Уровень значимости $\alpha = 0.05$.

In [36]:
e = np.array(results.resid)
n = len(e)

# оцениваем дисперсию остатков
sigma2_est = np.var(e, ddof=1)
sigma_est = np.sqrt(sigma2_est)

# считаем наблюдаемую статистику Колмогорова
# kstest возвращает D_n = sup |F_emp - F_teor|
D_n, _ = stats.kstest(e, 'norm', args=(0, sigma_est))
delta_obs = np.sqrt(n) * D_n

# параметрический бутстрап
N = 50000
m = 0

for _ in range(N):
    # генерируем подвыборку
    e_star = np.random.normal(0, sigma_est, n)

    # дисперсия для новой выборки
    sigma_star = np.std(e_star, ddof=1)

    D_star, _ = stats.kstest(e_star, 'norm', args=(0, sigma_star))
    delta_star = np.sqrt(n) * D_star

    if delta_star >= delta_obs:
        m += 1

p_value = m / N

print(f"n = {n}")
print(f"σ̃² = {sigma2_est:.4f}")
print(f"Δ̃ = {delta_obs:.4f}")
print(f"p-value = {p_value:.4f}\n")

alpha = 0.05
if p_value < alpha:
    print(f"Итог: p-value < {alpha}")
    print("H₀ отвергается. Распределение ошибок не является нормальным.")
else:
    print(f"Итог: p-value >= {alpha}")
    print("Нет оснований отвергать H. Ошибки распределены нормально.")

n = 50
σ̃² = 2.6429
Δ̃ = 0.8002
p-value = 0.4497

Итог: p-value >= 0.05
Нет оснований отвергать H. Ошибки распределены нормально.


## Исследование регрессии на выбросы
Boxplot:

1. Ищем 25-й ($q_1$) и 75-й ($q_3$) процентили для выборки $e$.
2. Вычисляем $IQR = q_3 - q_1$.
3. Границы:
   * Нижняя: $q_1 - 1.5 \cdot IQR$
   * Верхняя: $q_3 + 1.5 \cdot IQR$
4. Всё, что за границами - выбросы.


In [ ]:
e = np.array(results.resid)

q1 = np.percentile(e, 25)
q3 = np.percentile(e, 75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# индексы и значения выбросов
outliers_idx = np.where((e < lower_bound) | (e > upper_bound))[0]
outliers_val = e[outliers_idx]

print(f"q1 = {q1:.4f}")
print(f"q3 = {q3:.4f}")
print(f"IQR = {iqr:.4f}\n")

if len(outliers_idx) > 0:
    print(f"Найдено выбросов: {len(outliers_idx)} шт")
    for idx, val in zip(outliers_idx, outliers_val):
        print(f"   - Наблюдение {idx + 1}, остаток = {val:.4f}")
else:
    print("Выбросов нет")

import matplotlib.pyplot as plt


plt.figure(figsize=(10, 3))

plt.boxplot(e, vert=False, patch_artist=True,
            flierprops=dict(marker='o', markerfacecolor='red', markersize=8))

plt.title('Boxplot остатков', fontsize=14)
plt.xlabel('e', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## Кросс-проверка регрессии


**Алгоритм:**
1. Из выборки объема $n$ исключается одно $i$-е наблюдение.
2. По оставшимся $n-1$ наблюдениям заново строятся оценки параметров регрессии (МНК).
3. Полученная модель делает прогноз $\tilde{y}_i$ для исключенной точки $x_i$.
4. Вычисляется квадрат ошибки для этого прогноза:
   $$ CVSS_i = (y_i - \tilde{y}_i)^2 $$
5. Процедура повторяется для всех $n$ точек, после чего ошибки суммируются:
   $$ CVSS = \sum_{i=1}^n CVSS_i $$

**Метрика качества:**
Для оценки строится кросс-валидационный коэффициент детерминации:
$$ R^2_{cv} = \frac{TSS - CVSS}{TSS} $$

Значение $R^2_{cv}$ сравнивается с исходным $R^2$. Если $R^2_{cv}$ значительно меньше, это свидетельствует о переобучении.

In [ ]:
# exog - матрица факторов X (с добавленной колонкой единиц)
# endog - вектор откликов Y
X = results.model.exog
Y = results.model.endog
n = len(Y)

cvss = 0

for i in range(n):
    # выкидываем i-ю строку из данных
    X_train = np.delete(X, i, axis=0)
    Y_train = np.delete(Y, i)

    # строим модель на оставшихся n-1 точках
    model_cv = sm.OLS(Y_train, X_train).fit()

    # делаем прогноз для выкинутой i-й точки
    y_pred_i = np.dot(X[i], model_cv.params)

    cvss_i = (Y[i] - y_pred_i)**2
    cvss += cvss_i

tss = results.centered_tss
r2_cv = (tss - cvss) / tss
r2_original = results.rsquared

print(f"CVSS = {cvss:.4f}")
print(f"R² = {r2_original:.4f}")
print(f"R²_cv = {r2_cv:.4f}\n")

if r2_cv < 0:
    print("R²_cv отрицательный.")
elif (r2_original - r2_cv) > 0.2:
    print("R²_cv заметно ниже оригинального. Присутствует переобучение.")
else:
    print("Разница между R² и R²_cv невелика. Нет переобучения")

## Проверка адекватности регрессии

Для проверки адекватности фиксируем точку (x_k = 0) и проводим $m = 5$ повторных измерений.
Истинный закон распределения отклика задан как $\eta \sim N(2 + 3x_1 - 2x_2 + x_3 + x_4 - x_5, 1.5^2)$.
В точке $x_k = 0$ этот закон принимает вид $N(2, 1.5^2)$. Сгенерируем 5 значений из этого распределения.

**1. Вычисляем дисперсии:**
* Оценка дисперсии в этой точке: $\hat{\sigma}^2 = \frac{1}{m-1} \sum_{i=1}^m (y_i - \bar{y})^2$
* Остаточная дисперсия модели: $S_{res}^2 = \frac{RSS}{n-p}$

**2. Проверяем гипотезы:**
* $H_0: \sigma_{res}^2 = \sigma^2$ (Модель адекватна)
* $H_1: \sigma_{res}^2 > \sigma^2$

Статистика критерия: $\Delta = \frac{S_{res}^2}{\hat{\sigma}^2} \sim F(n-p, m-1)$.
Сравниваем полученное p-value с уровнем значимости $\alpha = 0.05$.

In [47]:
# 5 независимых измерений в точке x_k = 0
m = 5
true_mean = 2.0   # мат. ожидание при x=0
true_std = 1.5    # корень из дисперсии

y_new = np.random.normal(loc=true_mean, scale=true_std, size=m)
print(f"5 новых откликов: {np.round(y_new, 3)}")

# вычисляем выборочную дисперсию этих 5 измерений
sigma2_y = np.var(y_new, ddof=1)
print(f"σ̂² = {sigma2_y:.4f}")

# достаем остаточную дисперсию из нашей модели
n_p = results.df_resid          # n - p (степени свободы)
sigma2_res = results.scale      # RSS / (n-p) в statsmodels
print(f"S_res² = {sigma2_res:.4f}\n")

# статистика Фишера
F_stat = sigma2_res / sigma2_y
print(f"Δ = {F_stat:.4f}")

p_value = stats.f.sf(F_stat, n_p, m - 1)
print(f"p-value = {p_value:.4f}\n")

alpha = 0.05
if p_value < alpha:
    print(f"p-value < {alpha}")
    print("Гипотеза H₀ отвергается. Модель неадекватна.")
else:
    print(f"p-value >= {alpha}")
    print("Нет оснований отвергать H₀. Модель адекватна.")

5 новых откликов: [1.303 3.582 2.476 1.31  1.727]
σ̂² = 0.9338
S_res² = 2.9432

Δ = 3.1518
p-value = 0.1351

p-value >= 0.05
Нет оснований отвергать H₀. Модель адекватна.


## Упрощение регрессии
Находим фактор с наименьшей статистической значимостью (с наибольшим p-value из пункта b)

$H_0$: удаленный фактор не влияет на отклик, и усложнение регрессии незначимо.
Сравниваем две модели:
* **Полная модель:** $p_1$ параметров, ошибка $RSS_1$
* **Упрощенная модель:** $p_0 = p_1 - 1$ параметров, ошибка $RSS_0$

Для проверки используется статистика:
$$\Delta = \frac{(RSS_0 - RSS_1) / (p_1 - p_0)}{RSS_1 / (n - p_1)} \sim F(p_1 - p_0, n - p_1)$$
Если $p\text{-value} \ge 0.05$, принимаем $H_0$ и оставляем упрощенную модель. Сравниваем коэффициенты детерминации $R^2$ обеих моделей.

In [60]:
# находим фактор с максимальным p-value
p_values = results.pvalues[1:]
worst_idx_in_slice = np.argmax(p_values)
worst_idx = worst_idx_in_slice + 1
worst_p_val = p_values[worst_idx_in_slice]

print(f"Удаляем фактор №{worst_idx} (p-value = {worst_p_val:.4f})\n")

# строим упрощенную модель
X = results.model.exog
Y = results.model.endog
X_simplified = np.delete(X, worst_idx, axis=1)

results_simplified = sm.OLS(Y, X_simplified).fit()


# --------------------------------------------------------------------
# пропускаем удаленный фактор
labels = ['const (β0)', 'β1', 'β2', 'β3', 'β4', 'β5']
labels_simplified = [label for i, label in enumerate(labels) if i != worst_idx]

summary_df = pd.DataFrame({
    'β̃': results_simplified.params,
    'Δ̃': results_simplified.tvalues,
    'p-value': results_simplified.pvalues
}, index=labels_simplified)

alpha = 0.05
summary_df['Вывод'] = summary_df['p-value'].apply(
    lambda p: 'Значим' if p < alpha else 'Незначим'
)
display(summary_df.round(4))


# --------------------------------------------------------------------
r_squared_sim = results_simplified.rsquared
delta_tilde_sim = results_simplified.fvalue
p_value_f_sim = results_simplified.f_pvalue

print(f"R² = {r_squared_sim:.4f}")
print(f"Δ̃ = {delta_tilde_sim:.4f}")
print(f"p-value = {p_value_f_sim:.4e}")

if p_value_f_sim < alpha:
    print(f"Итог: p-value < {alpha}. Регрессия значима.\n")
else:
    print(f"Итог: p-value >= {alpha}. Регрессия незначима.\n")

# --------------------------------------------------------------------

print("Сравнение уравнений регрессии")
rss_1 = results.ssr
rss_0 = results_simplified.ssr
df_diff = results_simplified.df_resid - results.df_resid

# проводим F-тест для вложенных моделей
F_stat = ((rss_0 - rss_1) / df_diff) / (rss_1 / results.df_resid)
p_value_F = stats.f.sf(F_stat, df_diff, results.df_resid)

print(f"RSS полной модели:        {rss_1:.4f}")
print(f"RSS упрощенной модели:    {rss_0:.4f}")
print(f"F-статистика:             {F_stat:.4f}")
print(f"p-value:         {p_value_F:.4f}\n")

if p_value_F >= alpha:
    print(f"p-value >= {alpha}. Нет оснований отвергать H₀.")
    print(f"Удаление фактора №{worst_idx} не привело к значимому ухудшению.\n")
else:
    print(f"p-value < {alpha}. Гипотеза H₀ отвергается.\n")

print("Сравнение R² ")
print(f"R² полной:      {results.rsquared:.4f}")
print(f"R² упрощенной:  {results_simplified.rsquared:.4f}")
print(f"Падение R²:     {results.rsquared - results_simplified.rsquared:.5f}")

Удаляем фактор №4 (p-value = 0.0680)



,β̃,Δ̃,p-value,Вывод
const (β0),2.0714,8.1725,0.0000,Значим
β1,2.7487,6.1378,0.0000,Значим
β2,-2.0720,-4.5259,0.0000,Значим
β3,0.9761,2.1797,0.0346,Значим
β5,-1.0390,-2.3757,0.0218,Значим


R² = 0.5967
Δ̃ = 16.6418
p-value = 1.9356e-08
Итог: p-value < 0.05. Регрессия значима.

Сравнение уравнений регрессии
RSS полной модели:        129.4998
RSS упрощенной модели:    139.8042
F-статистика:             3.5011
p-value:         0.0680

p-value >= 0.05. Нет оснований отвергать H₀.
Удаление фактора №4 не привело к значимому ухудшению.

Сравнение R² 
R² полной:      0.6264
R² упрощенной:  0.5967
Падение R²:     0.02973


## Сравнение уравнений регрессии бутстрапом
Для проверки гипотезы о возможности упрощения регрессии используем непараметрический бутстрап.

Исследуем величину $h$ — истинную разность остаточных дисперсий упрощенной и полной моделей:
$$h = RSS_0 - RSS_1$$
* $H_0: h = 0$ (Удаленный фактор не влияет, усложнение незначимо).
* $H_1: h > 0$ (Удаление фактора значимо ухудшило модель).

Наблюдаемое значение из пункта 'j': $\tilde{h} = \widetilde{RSS}_0 - \widetilde{RSS}_1$.

**Алгоритм**
Генерируем $N = 1000$ бутстрап-выборок путем случайного выбора строк из исходной таблицы $(y_i, \vec{x}_i)$ с возвращением.
Для каждой итерации строим обе модели, находим $\tilde{h}^* = RSS_0^* - RSS_1^*$ и вычисляем отклонение:
$$\Delta^* = \tilde{h}^* - \tilde{h}$$

**Доверительный интервал**
По вариационному ряду $\Delta^*_{(1)} \le \dots \le \Delta^*_{(N)}$ строим доверительный интервал для $h$ с уровнем значимости $\alpha = 0.05$:
$$\tilde{h} - \Delta^*_{(N)} < h < \tilde{h} - \Delta^*_{(K)}$$
Где $K = [\alpha \cdot N]$.

Если $0$ попадает в этот интервал, нет оснований отвергать $H_0$. Модель можно упростить.

In [56]:
# считаем наблюдаемую разность
h_ = results_simplified.ssr - results.ssr

N = 1000
alpha = 0.05
deltas = []

data_full = pd.concat([pd.Series(np.asarray(Y), name='Y'), pd.DataFrame(np.asarray(X))], axis=1)

for _ in range(N):
    sample_data = data_full.sample(frac=1, replace=True)

    Y_star = sample_data.iloc[:, 0]
    X_star = sample_data.iloc[:, 1:]

    # упрощенная матрица (убираем наименее значимый фактор из пункта j)
    X_sim_star = X_star.drop(columns=[worst_idx], errors='ignore')

    # обучаем обе модели
    res1_star = sm.OLS(Y_star, X_star).fit()
    res0_star = sm.OLS(Y_star, X_sim_star).fit()

    h_star = res0_star.ssr - res1_star.ssr
    deltas.append(h_star - h_)

# вариационный ряд
deltas_sorted = np.sort(deltas)

K = int(alpha * N)

left_bound = h_ - deltas_sorted[-1]
right_bound = h_ - deltas_sorted[K - 1]

print(f"Наблюдаемая разность = {h_:.4f}")
print(f"Доверительный интервал для h: [{left_bound:.4f},  {right_bound:.4f}]\n")

if left_bound <= 0 <= right_bound:
    print("0 входит в доверительный интервал ")
    print("Гипотеза H₀ не отвергается: модель можно упростить.")
else:
    print("0 не входит в доверительный интервал")
    print("Гипотеза H₀ отвергается")

Наблюдаемая разность = 10.3044
Доверительный интервал для h: [-97.5619,  20.1423]

0 входит в доверительный интервал 
Гипотеза H₀ не отвергается: модель можно упростить.
